Предварительно про PyTorch:
* [Про тензоры в pytorch](https://colab.research.google.com/github/pytorch/tutorials/blob/gh-pages/_downloads/tensor_tutorial.ipynb)
* [Про автоматическое дифференцирование и что такое .backwards()](https://colab.research.google.com/github/pytorch/tutorials/blob/gh-pages/_downloads/autograd_tutorial.ipynb)
* [Очень простая нейронка на pytorch](https://colab.research.google.com/drive/1RsZvw4KBGn5U5Aj5Ak7OG2pHx6z1OSlF)

# Классификация текстов

## Fakenews

1. Мы будем работать с данными fakenews отсюда: https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/Constraint_Train.csv
2. Проведите препроцессинг текста. Разбейте данные на train и test для задачи классификации.
3. Векторизуйте.
4. Обучите на полученных векторах алгоритм классификации.

Мы уже видели как эта задача выполняется с помощью Word2vec. Давайте вспомним.

In [57]:
from google.colab import drive
drive.mount('/content/drive') # Монтируем Google Drive для доступа к файлам

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
# Базовый путь к данным в Google Drive
base_path = '/content/drive/MyDrive/netology.ru/NLP/'

In [59]:
import pandas as pd

In [60]:
df = pd.read_csv(base_path + 'Constraint_Train.csv')

In [61]:
df.head()

# id	tweet	label
# 0	1	The CDC currently reports 99031 deaths. In gen...	real
# 1	2	States reported 1121 deaths a small rise from ...	real
# 2	3	Politically Correct Woman (Almost) Uses Pandem...	fake
# 3	4	#IndiaFightsCorona: We have 1524 #COVID testin...	real
# 4	5	Populous states can generate large case counts...	real

,id,tweet,label
0,1,The CDC currently reports 99031 deaths. In gen...,real
1,2,States reported 1121 deaths a small rise from ...,real
2,3,Politically Correct Woman (Almost) Uses Pandem...,fake
3,4,#IndiaFightsCorona: We have 1524 #COVID testin...,real
4,5,Populous states can generate large case counts...,real


In [62]:
df.info()

# <class 'pandas.core.frame.DataFrame'>
# RangeIndex: 6420 entries, 0 to 6419
# Data columns (total 3 columns):
#  #   Column  Non-Null Count  Dtype
# ---  ------  --------------  -----
#  0   id      6420 non-null   int64
#  1   tweet   6420 non-null   object
#  2   label   6420 non-null   object
# dtypes: int64(1), object(2)
# memory usage: 150.6+ KB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6420 entries, 0 to 6419
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      6420 non-null   int64 
 1   tweet   6420 non-null   object
 2   label   6420 non-null   object
dtypes: int64(1), object(2)
memory usage: 150.6+ KB


In [63]:
from nltk.tokenize import word_tokenize
from tqdm import tqdm

In [64]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [65]:
sentences = [word_tokenize(text.lower()) for text in tqdm(df.tweet)]

100%|██████████| 6420/6420 [00:01<00:00, 5099.39it/s]


In [66]:
%%time
from gensim.models.word2vec import Word2Vec

# Обучаем модель Word2Vec
model_tweets = Word2Vec(sentences, workers=4, vector_size=300, min_count=3, window=5, epochs=15)

# CPU times: user 6.25 s, sys: 103 ms, total: 6.35 s
# Wall time: 4.99 s

CPU times: user 5.89 s, sys: 44.1 ms, total: 5.93 s
Wall time: 4.1 s


In [67]:
model_tweets.wv.most_similar('france')

# [('2015', 0.9192318320274353),
#  ('verses', 0.9186397790908813),
#  ('tower', 0.918623685836792),
#  ('bags', 0.9139482975006104),
#  ('front', 0.9128363132476807),
#  ('flour', 0.9114679098129272),
#  ('arrest', 0.9113063216209412),
#  ('floor', 0.9100168943405151),
#  ('spanish', 0.9093238115310669),
#  ('french', 0.9065400958061218)]

[('2015', 0.9452831745147705),
 ('representative', 0.9229007363319397),
 ('tower', 0.9223160743713379),
 ('prayers', 0.9143679738044739),
 ('jacksonville', 0.9116407036781311),
 ('2017', 0.9079797267913818),
 ('aires', 0.9074481129646301),
 ('floor', 0.9064769744873047),
 ('clinton', 0.9055445194244385),
 ('husband', 0.9038134813308716)]

In [68]:
import numpy as np

In [69]:
def get_text_embedding(text):
    result = []
    for word in word_tokenize(text.lower()):
        if word in model_tweets.wv:
            result.append(model_tweets.wv[word])

    if len(result):
        result = np.sum(result, axis=0)
    else:
        result = np.zeros(300)
    return result

In [70]:
features = [get_text_embedding(text) for text in tqdm(df.tweet)]

100%|██████████| 6420/6420 [00:01<00:00, 3908.88it/s]


In [71]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [72]:
X_train, X_test, y_train, y_test = train_test_split(features, df.label, test_size=0.33)

In [73]:
model = LogisticRegression()
model.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [74]:
from sklearn.metrics import classification_report

In [75]:
predicted = model.predict(X_test)

In [76]:
print(classification_report(y_test, predicted))

              precision    recall  f1-score   support

        fake       0.91      0.91      0.91       986
        real       0.92      0.92      0.92      1133

    accuracy                           0.92      2119
   macro avg       0.92      0.92      0.92      2119
weighted avg       0.92      0.92      0.92      2119



```
precision    recall  f1-score   support

        fake       0.91      0.91      0.91      1066
        real       0.91      0.91      0.91      1053

    accuracy                           0.91      2119
   macro avg       0.91      0.91      0.91      2119
weighted avg       0.91      0.91      0.91      2119
```

###  Что будет, если использовать самый наивный метод?

In [77]:
from sklearn.feature_extraction.text import CountVectorizer

In [78]:
vec = CountVectorizer()

In [79]:
bow = vec.fit_transform(df.tweet)

In [80]:
X_train, X_test, y_train, y_test = train_test_split(features, df.label, test_size=0.33)
model = LogisticRegression()
model.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [81]:
predicted = model.predict(X_test)
print(classification_report(y_test, predicted))

              precision    recall  f1-score   support

        fake       0.90      0.92      0.91       997
        real       0.93      0.91      0.92      1122

    accuracy                           0.92      2119
   macro avg       0.92      0.92      0.92      2119
weighted avg       0.92      0.92      0.92      2119



```
              precision    recall  f1-score   support

        fake       0.90      0.92      0.91      1018
        real       0.92      0.91      0.92      1101

    accuracy                           0.91      2119
   macro avg       0.91      0.91      0.91      2119
weighted avg       0.91      0.91      0.91      2119
```


Конечно, мы всегда можем поиграться с предобработкой.

### PyTorch + LSTM

In [82]:
labels = (df.label == 'real').astype(int).to_list()

Нужно заранее задать размер для макксимальной длины предложений.

In [83]:
token_lists = [word_tokenize(text.lower()) for text in df.tweet]
max_len = len(max(token_lists, key=len))

In [84]:
max_len
# 1592

1592

Это слишком много. Но какая длина обычно?

In [85]:
from collections import Counter
fd = Counter([len(tokens) for tokens in token_lists])

In [86]:
fd.most_common(10)
# [(20, 178), (25, 174), (22, 170), (18, 170), (19, 168), (21, 168), (16, 163), (17, 162), (15, 160), (23, 156)]

[(20, 178),
 (25, 174),
 (22, 170),
 (18, 170),
 (19, 168),
 (21, 168),
 (16, 163),
 (17, 162),
 (15, 160),
 (23, 156)]

[(20, 178),
 (25, 174),
 (22, 170),
 (18, 170),
 (19, 168),
 (21, 168),
 (16, 163),
 (17, 162),
 (15, 160),
 (23, 156)]

Зададим максимум 200.

Возьмём те же w2v эмбеддинги.

In [87]:
def get_word_embedding(tokens, max_len):
    result = []
    for i in range(max_len):
        if i < len(tokens):
            word = tokens[i]
            if word in model_tweets.wv:
                result.append(model_tweets.wv[word])
            else:
                result.append(np.zeros(300))
        else:
            result.append(np.zeros(300))
    return result

In [88]:
features = [get_word_embedding(text, 200) for text in tqdm(token_lists)]

100%|██████████| 6420/6420 [00:00<00:00, 6652.87it/s]


In [89]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.33)

In [90]:
import torch
import torch.nn as nn
import torch.optim as optim

In [91]:
len(features[0][0])
# 300

300

In [92]:
len(X_train)
# 4301

4301

In [93]:
len(X_train[0])
# 200

200

In [94]:
len(X_train[0][0])
# 300

300

In [95]:
class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()
        self.lstm = nn.LSTM(300, 100)
        self.out = nn.Linear(100, 1)

    def forward(self, x):
        embeddings, (shortterm, longterm) = self.lstm(x.transpose(0, 1))
        prediction = torch.sigmoid(self.out(longterm))
        return prediction


net = Net()
print(net)

#Net(
#  (lstm): LSTM(300, 100)
 # (out): Linear(in_features=100, out_features=1, bias=True)
#)

Net(
  (lstm): LSTM(300, 100)
  (out): Linear(in_features=100, out_features=1, bias=True)
)


In [96]:
print(type(X_train))  # Если это список, то нужно преобразовать в массив
# <class 'list'>

<class 'list'>


In [97]:
# Преобразуем список массивов в один массив NumPy
X_train_np = np.array(X_train)

In [98]:
print(type(X_train_np))
# <class 'numpy.ndarray'>

<class 'numpy.ndarray'>


In [99]:
in_data = torch.tensor(X_train_np).float()
targets = torch.tensor(y_train).float()

In [100]:
in_data.shape
# torch.Size([4301, 200, 300])

torch.Size([4301, 200, 300])

In [101]:
optimizer = optim.SGD(net.parameters(), lr=0.01)
criterion = nn.BCELoss()

In [102]:
def train_one_epoch(in_data, targets, batch_size=16):
    for i in tqdm(range(0, in_data.shape[0], batch_size)):
        batch_x = in_data[i:i + batch_size]
        batch_y = targets[i:i + batch_size]
        optimizer.zero_grad()
        output = net(batch_x)
        loss = criterion(output.reshape(-1), batch_y)
        loss.backward()
        optimizer.step()
    print(loss)

In [103]:
train_one_epoch(in_data, targets)

100%|██████████| 269/269 [00:15<00:00, 16.99it/s]

tensor(0.6965, grad_fn=<BinaryCrossEntropyBackward0>)


Что получилось?

In [104]:
print(type(X_test))  # Если это список, то нужно преобразовать в массив

# <class 'list'>

<class 'list'>


In [105]:
# Преобразуем список массивов в один массив NumPy
X_test_np = np.array(X_test)

In [106]:
in_data_test = torch.tensor(X_test_np).float()
targets_test = torch.tensor(y_test).float()

In [107]:
with torch.no_grad():
    output = net(in_data_test).reshape(-1)

In [108]:
result = (output > 0.5) == targets_test

In [109]:
result.sum().item() / len(result)

# 0.5224162340726758

0.5176970268994809

Но такую модель надо учить дольше(